# Линейная регрессия — мини-проект

Сквозной мини-проект: один датасет, решения передаются между этапами.


In [ ]:
# Setup
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print
import warnings
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.linear_model import HuberRegressor

np.random.seed(42)



## Постановка задачи `(intro)`


Предскажем **медианную стоимость жилья** в Калифорнии (OpenML `california_housing`) по демографическим и географическим признакам.

**Сюжет (как в продакшн):** загрузка → feature engineering → **OHE `ocean_proximity`** → EDA → split → IQR на train → подбор `alpha` → IQR vs **Huber** → метрики на полном test.


## Загрузка данных `(eda)`


In [ ]:
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", font_scale=1.05)

raw = fetch_openml(name="california_housing", version=1, as_frame=True, parser="auto")
df = raw.frame.copy()
target_col = "median_house_value"
df = df.fillna(df.median(numeric_only=True))  # редкие NaN в OpenML; дальше — только train-статистики

print(f"Объектов: {len(df)}, столбцов: {len(df.columns)}")
display(df.head())


## Feature engineering `(example)`


Сырые счётчики (`total_rooms`, `population`, …) сильно коррелируют. Заменяем их **удельными метриками** — модель остаётся линейной, но признаки информативнее.


In [ ]:
df["rooms_per_household"] = df["total_rooms"] / df["households"]
df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"].replace(0, np.nan)
df["population_per_household"] = df["population"] / df["households"]

feature_cols = [
    "housing_median_age",
    "median_income",
    "latitude",
    "longitude",
    "rooms_per_household",
    "bedrooms_per_room",
    "population_per_household",
]
cat_col = "ocean_proximity"
model_cols = feature_cols + [cat_col]

print(f"Числовых: {len(feature_cols)}, категория: {cat_col}")
print(df[cat_col].value_counts())
display(df[feature_cols].describe().round(3))


## Категория: ocean_proximity `(example)`


Расстояние до океана — **категориальный** признак. В линейной модели — **One-Hot Encoding** внутри `ColumnTransformer`: числовые столбцы масштабируем, категории кодируем (`drop='first'`).


In [ ]:
ocean_price = df.groupby(cat_col)[target_col].median().sort_values(ascending=False)
print("Медиана цены по району:")
display(ocean_price.to_frame("median_house_value"))


## EDA: распределения и корреляции `(viz)`


Сильнейший сигнал — `median_income`. На гистограмме цены — **хвосты** (пороги IQR посчитаем после split, только на train). Удельные признаки снижают корреляции между столбцами.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df[target_col], bins=40, color="steelblue", edgecolor="white")
axes[0].set_xlabel("median_house_value")
axes[0].set_title("Распределение цены")

axes[1].scatter(df["median_income"], df[target_col], alpha=0.25, s=12, color="steelblue")
axes[1].set_xlabel("median_income")
axes[1].set_ylabel("median_house_value")
axes[1].set_title("Цена vs доход")

corr = df[feature_cols + [target_col]].corr()
sns.heatmap(corr, ax=axes[2], cmap="RdBu_r", center=0, vmin=-1, vmax=1)
axes[2].set_title("Корреляции")
plt.tight_layout()
plt.show()

print("Корр. rooms_per_household–bedrooms_per_room:",
      corr.loc["rooms_per_household", "bedrooms_per_room"].round(3))
print("Корр. median_income–цена:", corr.loc["median_income", target_col].round(3))


## Решение: train / test split `(example)`


**Решение (продакшн):** сначала **один** split на сыром датасете (`random_state=42`). Любые пороги, imputation и scaler — **только из train**, test не участвует.


In [ ]:
idx_all = np.arange(len(df))
idx_train, idx_test = train_test_split(idx_all, test_size=0.2, random_state=42)

# imputation: медианы только по train
feat_median = df.iloc[idx_train][feature_cols].median()
df[feature_cols] = df[feature_cols].fillna(feat_median)

X_train_full = df.iloc[idx_train][model_cols]
y_train_full = df.iloc[idx_train][target_col].values
X_test = df.iloc[idx_test][model_cols]
y_test = df.iloc[idx_test][target_col].values

print(f"Train: {len(idx_train)}, test: {len(idx_test)}")


## Выбросы: IQR только на train `(experiment)`


MSE сильно штрафует экстремальные **Y**. Правило **IQR** считаем по **`y_train`**, границы фиксируем и применяем к train/test — без подглядывания в test.


In [ ]:
y_train_series = pd.Series(y_train_full)
q1, q3 = y_train_series.quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = float(q1 - 1.5 * iqr), float(q3 + 1.5 * iqr)

train_in_bounds = (y_train_full >= lower) & (y_train_full <= upper)
test_in_bounds = (y_test >= lower) & (y_test <= upper)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(y=y_train_series, ax=axes[0], color="steelblue")
axes[0].axhline(lower, color="crimson", linestyle="--", lw=1, label="train IQR")
axes[0].axhline(upper, color="crimson", linestyle="--", lw=1)
axes[0].set_title("Boxplot y_train")
axes[0].set_ylabel("median_house_value")
axes[0].legend(fontsize=8)

income_tr = df.iloc[idx_train]["median_income"].values
axes[1].scatter(income_tr[train_in_bounds], y_train_full[train_in_bounds],
                alpha=0.2, s=10, color="steelblue", label="train, в границах")
axes[1].scatter(income_tr[~train_in_bounds], y_train_full[~train_in_bounds],
                alpha=0.8, s=25, color="crimson", label="train, выброс IQR")
axes[1].set_xlabel("median_income")
axes[1].set_ylabel("median_house_value")
axes[1].set_title("Выбросы train по IQR(train)")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"IQR(train): [{lower:,.0f}; {upper:,.0f}]")
print(f"Выбросов в train: {(~train_in_bounds).sum()} ({100 * (~train_in_bounds).mean():.1f}%)")
print(f"Test вне границ train-IQR: {(~test_in_bounds).sum()} ({100 * (~test_in_bounds).mean():.1f}%)")


## Решение: очистка train `(example)`


**Решение:** из **train** убираем объекты вне `[lower, upper]`. **Test не фильтруем** — в продакшн модель должна отвечать на любой объект; метрики считаем на **полном** hold-out.


In [ ]:
idx_train_clean = idx_train[train_in_bounds]
X_train = df.iloc[idx_train_clean][model_cols]
y_train = df.iloc[idx_train_clean][target_col].values

print(f"Train до IQR: {len(idx_train)} → после: {len(idx_train_clean)}")
print(f"Test без изменений: {len(idx_test)} объектов")


## Baseline: OLS в Pipeline `(example)`


`ColumnTransformer`: числовые → `StandardScaler`, `ocean_proximity` → `OneHotEncoder` — fit **только на train**.


In [ ]:
def make_preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), feature_cols),
        ("cat", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"), [cat_col]),
    ])

def make_model_pipe(model):
    return Pipeline([
        ("prep", make_preprocessor()),
        ("model", model),
    ])

pipe_ols = make_model_pipe(LinearRegression())
pipe_ols.fit(X_train, y_train)
y_pred_ols = pipe_ols.predict(X_test)

print(f"OLS — Test MAE:  {mean_absolute_error(y_test, y_pred_ols):,.0f}")
print(f"OLS — Test RMSE: {mean_squared_error(y_test, y_pred_ols) ** 0.5:.3f}")
print(f"OLS — Test R2:   {r2_score(y_test, y_pred_ols):.3f}")
print("Признаков после OHE:", len(pipe_ols.named_steps["prep"].get_feature_names_out()))


## Эксперимент: log1p(y) `(experiment)`


На EDA видны **хвосты** цены — классический приём: обучать модель на `log1p(y)`, предсказания возвращать через `expm1`. Проверим, даёт ли `log1p(y)` выигрыш на **очищенном train** (пороги IQR — только с train).


In [ ]:
def make_regressor(model, log_target=False):
    pipe = make_model_pipe(model)
    if log_target:
        return TransformedTargetRegressor(
            regressor=pipe,
            func=np.log1p,
            inverse_func=np.expm1,
        )
    return pipe

def model_coef(estimator):
    if isinstance(estimator, TransformedTargetRegressor):
        return estimator.regressor_.named_steps["model"].coef_
    return estimator.named_steps["model"].coef_

def fitted_preprocessor(estimator):
    if isinstance(estimator, TransformedTargetRegressor):
        return estimator.regressor_.named_steps["prep"]
    return estimator.named_steps["prep"]

log_compare = []
for log_y in (False, True):
    m = make_regressor(LinearRegression(), log_target=log_y)
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    log_compare.append({
        "log1p(y)": log_y,
        "R2 test (исходная шкала)": r2_score(y_test, pred),
        "RMSE test": mean_squared_error(y_test, pred) ** 0.5,
    })

log_df = pd.DataFrame(log_compare)
display(log_df.round({"R2 test (исходная шкала)": 4, "RMSE test": 1}))

USE_LOG_Y = log_df.loc[log_df["R2 test (исходная шкала)"].idxmax(), "log1p(y)"]
label = "log1p(y)" if USE_LOG_Y else "исходная y"
print(f"**Решение:** для дальнейших шагов оставляем {label}.")


## Подбор alpha: Ridge, Lasso и ElasticNet `(experiment)`


`alpha` (и для ElasticNet — `l1_ratio`) подбираем **только на IQR-train** через `GridSearchCV` (5-fold). Test не участвует.


In [ ]:
alphas = np.logspace(-2, 3, 25)
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]

if USE_LOG_Y:
    ridge_base = TransformedTargetRegressor(
        regressor=make_model_pipe(Ridge()),
        func=np.log1p,
        inverse_func=np.expm1,
    )
    lasso_base = TransformedTargetRegressor(
        regressor=make_model_pipe(Lasso(max_iter=10000)),
        func=np.log1p,
        inverse_func=np.expm1,
    )
    enet_base = TransformedTargetRegressor(
        regressor=make_model_pipe(ElasticNet(max_iter=10000)),
        func=np.log1p,
        inverse_func=np.expm1,
    )
    ridge_cv = GridSearchCV(
        ridge_base,
        param_grid={"regressor__model__alpha": alphas},
        cv=5,
        scoring="neg_root_mean_squared_error",
    )
    lasso_cv = GridSearchCV(
        lasso_base,
        param_grid={"regressor__model__alpha": alphas},
        cv=5,
        scoring="neg_root_mean_squared_error",
    )
    enet_cv = GridSearchCV(
        enet_base,
        param_grid={
            "regressor__model__alpha": alphas,
            "regressor__model__l1_ratio": l1_ratios,
        },
        cv=5,
        scoring="neg_root_mean_squared_error",
    )
else:
    ridge_cv = GridSearchCV(
        make_model_pipe(Ridge()),
        param_grid={"model__alpha": alphas},
        cv=5,
        scoring="neg_root_mean_squared_error",
    )
    lasso_cv = GridSearchCV(
        make_model_pipe(Lasso(max_iter=10000)),
        param_grid={"model__alpha": alphas},
        cv=5,
        scoring="neg_root_mean_squared_error",
    )
    enet_cv = GridSearchCV(
        make_model_pipe(ElasticNet(max_iter=10000)),
        param_grid={"model__alpha": alphas, "model__l1_ratio": l1_ratios},
        cv=5,
        scoring="neg_root_mean_squared_error",
    )

ridge_cv.fit(X_train, y_train)
lasso_cv.fit(X_train, y_train)
enet_cv.fit(X_train, y_train)

if USE_LOG_Y:
    best_ridge_alpha = ridge_cv.best_params_["regressor__model__alpha"]
    best_lasso_alpha = lasso_cv.best_params_["regressor__model__alpha"]
    best_enet_alpha = enet_cv.best_params_["regressor__model__alpha"]
    best_enet_l1 = enet_cv.best_params_["regressor__model__l1_ratio"]
else:
    best_ridge_alpha = ridge_cv.best_params_["model__alpha"]
    best_lasso_alpha = lasso_cv.best_params_["model__alpha"]
    best_enet_alpha = enet_cv.best_params_["model__alpha"]
    best_enet_l1 = enet_cv.best_params_["model__l1_ratio"]

print(f"Ridge — alpha: {best_ridge_alpha:.4f}, CV RMSE: {-ridge_cv.best_score_:.3f}")
print(f"Lasso — alpha: {best_lasso_alpha:.4f}, CV RMSE: {-lasso_cv.best_score_:.3f}")
print(f"ElasticNet — alpha: {best_enet_alpha:.4f}, l1_ratio: {best_enet_l1}, CV RMSE: {-enet_cv.best_score_:.3f}")


## Сравнение: IQR-путь vs Huber `(experiment)`


IQR-путь: OLS / Ridge / Lasso / **ElasticNet** на **очищенном train**. **Huber** — на **полном train**. Метрики на **полном test**; для Huber смотрим и **MAE** (робастная модель ближе к модулю, чем к MSE).


In [ ]:
candidates = {
    "OLS (IQR train)": make_regressor(LinearRegression(), log_target=USE_LOG_Y),
    "Ridge (IQR train)": make_regressor(Ridge(alpha=best_ridge_alpha), log_target=USE_LOG_Y),
    "Lasso (IQR train)": make_regressor(
        Lasso(alpha=best_lasso_alpha, max_iter=10000), log_target=USE_LOG_Y
    ),
    "ElasticNet (IQR train)": make_regressor(
        ElasticNet(alpha=best_enet_alpha, l1_ratio=best_enet_l1, max_iter=10000),
        log_target=USE_LOG_Y,
    ),
}

rows = []
for name, pipe in candidates.items():
    pipe.fit(X_train, y_train)
    pred_te = pipe.predict(X_test)
    pred_in = pred_te[test_in_bounds]
    w = model_coef(pipe)
    rows.append({
        "model": name,
        "R2 test (full)": r2_score(y_test, pred_te),
        "R2 test (in bounds)": r2_score(y_test[test_in_bounds], pred_in),
        "MAE test": mean_absolute_error(y_test, pred_te),
        "RMSE test": mean_squared_error(y_test, pred_te) ** 0.5,
        "norm_w": np.linalg.norm(w),
    })

pipe_huber = make_model_pipe(HuberRegressor(max_iter=500))
pipe_huber.fit(X_train_full, y_train_full)
pred_huber = pipe_huber.predict(X_test)
pred_huber_in = pred_huber[test_in_bounds]
w_huber = pipe_huber.named_steps["model"].coef_
rows.append({
    "model": "Huber (full train)",
    "R2 test (full)": r2_score(y_test, pred_huber),
    "R2 test (in bounds)": r2_score(y_test[test_in_bounds], pred_huber_in),
    "MAE test": mean_absolute_error(y_test, pred_huber),
    "RMSE test": mean_squared_error(y_test, pred_huber) ** 0.5,
    "norm_w": np.linalg.norm(w_huber),
})

metrics_df = pd.DataFrame(rows).sort_values("RMSE test")
display(metrics_df.round(4))


## Решение: выбор модели `(example)`


**Решение:** берём модель с лучшим RMSE на **полном test**. Дальше все графики — на том же hold-out.


In [ ]:
best_name = metrics_df.iloc[0]["model"]
if best_name == "Huber (full train)":
    final_pipe = pipe_huber
else:
    final_pipe = candidates[best_name]
    final_pipe.fit(X_train, y_train)
y_pred = final_pipe.predict(X_test)
y_pred_train = final_pipe.predict(X_train)
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred)

print(f"Выбрана модель: {best_name}")
print(f"Train R2: {r2_train:.3f}  |  Test R2: {r2_test:.3f}")
if r2_train - r2_test > 0.05:
    print("→ Train >> Test: возможное переобучение")
print(f"Test MAE:  {mean_absolute_error(y_test, y_pred):,.0f}")
print(f"Test RMSE: {mean_squared_error(y_test, y_pred) ** 0.5:.3f}")


## Диагностика: predicted vs actual и остатки `(viz)`


**Predicted vs actual:** точки у диагонали — хороший прогноз. **Residual plot:** случайное облако вокруг нуля; «воронка» — нелинейность.


In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(y_test, y_pred, alpha=0.35, s=15, color="steelblue")
lo, hi = y_test.min(), y_test.max()
axes[0].plot([lo, hi], [lo, hi], "k--", lw=1, label="идеал ŷ = y")
axes[0].set_xlabel("факт y")
axes[0].set_ylabel("предсказание ŷ")
axes[0].set_title(f"Predicted vs actual ({best_name})")
axes[0].legend(fontsize=8)

axes[1].scatter(y_pred, residuals, alpha=0.35, s=15, color="steelblue")
axes[1].axhline(0, color="crimson", linestyle="--")
axes[1].set_xlabel("предсказание ŷ")
axes[1].set_ylabel("остаток y − ŷ")
axes[1].set_title("Residual plot")

axes[2].hist(residuals, bins=35, color="steelblue", edgecolor="white")
axes[2].set_xlabel("остаток")
axes[2].set_title("Гистограмма остатков")
plt.tight_layout()
plt.show()


## Интерпретация весов `(viz)`


После масштабирования сравниваем силу признаков (**ceteris paribus** — при прочих равных).


In [ ]:
w = model_coef(final_pipe)
feat_names = fitted_preprocessor(final_pipe).get_feature_names_out()
order = np.argsort(np.abs(w))

plt.figure(figsize=(9, 6))
plt.barh(np.array(feat_names)[order], w[order], color="steelblue")
plt.xlabel("вес (OHE + StandardScaler → модель)")
plt.title(f"Интерпретация весов ({best_name})")
plt.tight_layout()
plt.show()

for name, val in sorted(zip(feat_names, w), key=lambda t: abs(t[1]), reverse=True)[:5]:
    print(f"{name:28s}: {val:+.3f}")


## Чек-лист мини-проекта `(summary)`


1. **Split первым** — затем imputation, IQR и scaler только по train.
2. Feature engineering и категориальные признаки в `ColumnTransformer`.
3. Test **не фильтруем**; метрики на полном hold-out.
4. Ridge / Lasso / **ElasticNet** — `GridSearchCV` без участия test.
5. Сравнили IQR-путь и **Huber**; смотрели R², **MAE**, RMSE.
6. **Train vs test R²**, predicted vs actual, residual plot, интерпретация весов.


## Исследовательские вопросы `(summary)`


Дополнительные задания для **самостоятельной** работы. Сохраняйте каркас: split → IQR по train → `ColumnTransformer`, оценка на **полном** hold-out test. Фиксируйте **R², MAE, RMSE** и краткий вывод.

1. **Целевая `log1p(y)` vs исходная `y`:** при том же препроцессинге и IQR на train сравните метрики на исходной шкале цены (через `expm1`). Когда log-трансформация целевой оправдана, а когда нет?

2. **Порядок IQR и split:** (A) IQR на всём датасете → удаление → split; (B) split → IQR только на train. Чем отличаются R² и RMSE на test? Есть ли утечка в варианте A?

3. **Полный test vs in-bounds:** обучите модель на clean train, оцените на (a) всём test и (b) объектах внутри train-IQR. Какой протокол отражает поведение модели «в проде»?

4. **`log1p(median_income)` как признак:** замените или добавьте к raw income. Меняется ли качество линейной модели с OHE?

5. **`PolynomialFeatures(degree=2)` для `median_income`:** внутри `ColumnTransformer` + Ridge с подбором `alpha`. Окупается ли усложнение по метрикам и residual plot?

6. **Без `ocean_proximity`:** уберите OHE и сравните с базовым пайплайном. Как меняются R² и вклад lat/lon, income в весах?

7. **Сырые счётчики vs удельные метрики:** `total_rooms`, `population`, `households` вместо `rooms_per_household` и др. Как влияет мультиколлинеарность на `norm_w`?

8. **Стратегии для выбросов в `y`:** (a) IQR-trim train, (b) полный train + Huber, (c) полный train + OLS. При какой метрике — **MAE** или **RMSE** — каждый подход сильнее?

9. **Winsorize вместо удаления:** ограничьте `y` train-квантилями (например, 1–99%) без удаления строк. Сравните размер train, R² и MAE с IQR-trim.

10. **`QuantileRegressor`:** линейная модель с pinball-loss (медиана). Сравните с Huber и OLS по **MAE**; на predicted vs actual отметьте участки с наибольшими ошибками (дешёвое / дорогое жильё).
